In [0]:
%python
from pyspark.sql import functions as F
from delta.tables import DeltaTable

def write_to_silver(input_df, target_table, merge_condition, columns_to_update):
  """
  This function creates the delta table if it does not exist. Otherwise merges the input DataFrame into the target table. 
  
  Parameters:
  input_df: The input dataframe to be written to the target table.
  target_table: The name of the target table in the silver layer.
  merge_condition: The condition used for merging the input dataframe with the target table.
  columns_to_update: The columns to be updated in the target table.
  
  Returns:
  None
  """
    final_df=(input_df.withColumn("created_timestamp", F.current_timestamp())
                    .withColumn("updated_timestamp", F.current_timestamp())
    )

    if not spark.catalog.tableExists(target_table):
        (
          final_df.write
          .format("delta")
          .mode("overwrite")
          .saveAsTable(target_table)
        )
    else:
        delta_table = DeltaTable.forName(spark, target_table)
        update_map = {column: f"s.{column}" for column in columns_to_update}
        update_map["updated_timestamp"] = "s.updated_timestamp"
        (
          delta_table.alias("t")
          .merge(
            source=final_df.alias("s"),
            condition=merge_condition
          )
          .whenMatchedUpdate(condition="s.batch_id>t.batch_id", set=update_map)
          .whenNotMatchedInsertAll()
          .execute()
        )
  
